<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/05_evaluate_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade transformers
!pip install sentencepiece tiktoken
!pip install rank_bm25
!pip install sentence-transformers

In [ ]:
import os, sys
import torch
from google.colab import drive
from google.colab import userdata
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)
from vector_store import RailVectorVault
from embed import RailEmbedder
from evaluate import RailAuditJudge
from call_local_llm import run_integrated_audit

# Mount Drive to access Vector DB and save the model
drive.mount('/content/drive')
api_key = userdata.get('GEMINI_API_KEY')
DB_PATH = "/content/drive/MyDrive/rail_ai_project/vector_db"
model_name = "unsloth/Llama-3.2-3B-Instruct"
adapter_path = "/content/drive/MyDrive/rail_ai_project/rail_safety_adapters"

In [ ]:
api_key = userdata.get('GEMINI_API_KEY_TEMP')

In [ ]:

import importlib
import evaluate
importlib.reload(evaluate)
from evaluate import RailAuditJudge


In [ ]:
embedder = RailEmbedder(model_name='BAAI/bge-base-en-v1.5')
vault = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.for_inference(model)
model.load_adapter(adapter_path)

In [ ]:
model_id = "gemini-2.5-flash"

url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent"

In [ ]:
evaluator = RailAuditJudge(audit_function = run_integrated_audit,
                           model = model,
                           tokenizer = tokenizer,
                           vault = vault,
                           api_key = api_key,
                           judge_url = url)

In [ ]:
eval_set_path = '/content/rail-safety-ai/data/training/retriever_eval_set_1000_from_csv.json'

In [ ]:
import json
from collections import defaultdict

num_samples = 1
use_dynamic_filter = True

with open(eval_set_path, 'r') as f:
    eval_data = json.load(f)[:num_samples]

# Group samples by their source file for efficient batching
grouped_data = defaultdict(list)
for item in eval_data:
    source_file = item.get('file', 'Unknown') if use_dynamic_filter else None
    grouped_data[source_file].append(item)

results = []

In [ ]:
batch_size = 4


for source_file, items in grouped_data.items():
    print(f"\n Evaluating {len(items)} samples for source: {source_file or 'ALL (Unfiltered)'}")

    for i in range(0, len(items), batch_size):
        batch = items[i : i + batch_size]
        questions = [item['question'] for item in batch]

        # Inference with the specific source_filter for this manual
        batch_responses = run_integrated_audit(
            questions,
            vault,
            tokenizer,
            model,
            method='rerank',
            n_results=5,
            source_filter=source_file,
            show_context=False
        )

        for idx, raw_output in enumerate(batch_responses):
            item = batch[idx]
            ground_truth = item.get('answer_chunk', "N/A")

            try:
                parts = raw_output.split("[ANSWER]")
                thinking = parts[0].replace("[THINKING PROCESS]", "").strip()
                answer = parts[-1].strip()
            except Exception:
                thinking, answer = "Parse Error", raw_output


In [ ]:
question = items[0]["question"]
judge_prompt = f"""
        You are an expert FRA Safety Auditor. Grade the following AI-generated safety response against the Ground Truth reference.

        [STIMULUS]
        Question: {question}
        Ground Truth Reference (The True Manual Text): {ground_truth}

        [AI RESPONSE]
        Thinking Process (What the AI considered): {thinking}
        Final Answer: {answer}

        [RUBRIC]
        1. FAITHFULNESS (1-5): Is the answer derived ONLY from the context provided in the thinking process? (1 = Hallucinated/Used external knowledge, 5 = Perfectly Grounded)
        2. REGULATORY ACCURACY (1-5): Compare the AI Answer to the Ground Truth Reference. Does the logic match? (1 = Dangerous/Incorrect, 5 = Expert accuracy)
        3. CITATION QUALITY (1-5): Did the model cite specific Pages/Sections correctly as per the Thinking Process?

        Provide your critique and scores in a structured JSON format.
        """
payload = {
    "contents": [{"parts": [{"text": judge_prompt}]}],
    "generationConfig": {
        "responseMimeType": "application/json",
        "responseSchema": {
            "type": "OBJECT",
            "properties": {
                "faithfulness": {"type": "NUMBER"},
                "accuracy": {"type": "NUMBER"},
                "citation": {"type": "NUMBER"},
                "critique": {"type": "STRING"}
            },
            "required": ["faithfulness", "accuracy", "citation", "critique"]
        }
    }
}

In [ ]:
import requests
response = requests.post(f"{url}?key={api_key}", json=payload)

In [ ]:
print(response)

In [ ]:

report = evaluator.run_benchmark(eval_set_path = eval_set_path, num_samples=1, use_dynamic_filter=True)

In [ ]:
report